# PHASE 5
# EXPLAINABLE LAYER FOR BraTS2021
# Retrieval Augmented Explanations of the Findings

The structured findings produced in Phase 4 are the main output of the pipeline. This notebook adds an **explanation layer** that helps users understand those findings. When requested, a local language model explains the meaning of each result using a curated collection of open access imaging references.

The knowledge base is limited to imaging information and does not include histopathology, prognosis, or treatment. Each explanation is based on the most relevant reference passages and includes source citations. This supports the application's **"Learn more"** feature.

This complements the **provenance** information from Phase 4, which shows where each result came from, by also explaining what the findings mean with supporting references.

## Overview

1. **Reference library**:
   Load a curated collection of open access imaging references and index them for efficient retrieval.

2. **Retrieval augmented explanations**:
   Retrieve the most relevant reference passages for each finding and use a local language model to generate an explanation based only on those sources with citations included.

3. **Safety**:
   Generate descriptive explanations only, without providing diagnosis, prognosis or treatment advice.

The language model runs **locally** through Ollama, so no data is exposed.


## 1. Imports, configuration, and paths

In [2]:
import os
import json
import glob
import copy
import re
import numpy as np
import requests
%matplotlib inline
from sentence_transformers import SentenceTransformer
import faiss

In [3]:
# Configuration - local model backend (privacy-compliant)
# The explainer runs entirely on a local open-weight model via Ollama, so no patient-derived data leaves the machine.
LOCAL_MODEL = "llama3:8b"     # must be pulled:  ollama pull llama3:8b

TEMPERATURE = 0.2
REQUEST_TIMEOUT = 60            

# Retrieval settings
TOP_K = 3          # how many reference passages to retrieve per finding

print(f"Local model: {LOCAL_MODEL}")

Local model: llama3:8b


In [4]:
# Paths
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

# Phase 4 findings (structured findings JSON, one per patient)
findings_dir = os.path.join(project_root, "outputs", "description_outputs", "structured_findings")

# Curated open-access reference passages for retrieval (.txt files)
references_dir = os.path.join(project_root, "app", "references")

print("findings_dir exists:", os.path.isdir(findings_dir))
print("references_dir exists:", os.path.isdir(references_dir))

findings_dir exists: True
references_dir exists: True


## 2. Functions: retrieval-augmented explanation

In [5]:
# Retrieval-augmented explanation: define the helper functions that implement the explainer.

# module-level state
_MODEL = None
_INDEX = None
_CHUNKS = None
_EMBED_DIM = None

# load reference passages from .txt files
def _read_reference_files(ref_dir):
    """Read every .txt in ref_dir -> list of {text, source, file}.
    First line 'SOURCE: ...' is the citation; blank-line-separated paragraphs become chunks."""
    chunks = []
    for path in sorted(glob.glob(os.path.join(ref_dir, "*.txt"))):
        with open(path, encoding="utf-8") as f:
            raw = f.read().strip()
        source, body = "unknown source", raw
        if raw.lower().startswith("source:"):
            first, _, rest = raw.partition("\n")
            source = first.split(":", 1)[1].strip()
            body = rest.strip()
        for para in re.split(r"\n\s*\n", body):
            para = para.strip()
            if len(para) >= 40:
                chunks.append({"text": para, "source": source, "file": os.path.basename(path)})
    return chunks

# Build the FAISS embedding index
def build_index(ref_dir):
    global _MODEL, _INDEX, _CHUNKS, _EMBED_DIM
    _CHUNKS = _read_reference_files(ref_dir)
    if not _CHUNKS:
        return False
    try:
        _MODEL = SentenceTransformer("all-MiniLM-L6-v2")
        embs = _MODEL.encode([c["text"] for c in _CHUNKS],
                             convert_to_numpy=True, normalize_embeddings=True)
        _EMBED_DIM = embs.shape[1]
        _INDEX = faiss.IndexFlatIP(_EMBED_DIM)
        _INDEX.add(embs.astype("float32"))
        return True
    except Exception as e:
        print(f"[rag] embeddings/FAISS unavailable ({e}).")
        _MODEL = None; _INDEX = None
        return False

# retrieve top-k passages for a query
def retrieve(query, k=3):
    if _CHUNKS is None:
        return []
    if _MODEL is not None and _INDEX is not None:
        q = _MODEL.encode([query], convert_to_numpy=True, normalize_embeddings=True)
        scores, idx = _INDEX.search(q.astype("float32"), min(k, len(_CHUNKS)))
        return [_CHUNKS[i] for i in idx[0] if i >= 0]
    qwords = set(re.findall(r"[a-z]{4,}", query.lower()))
    scored = []
    for c in _CHUNKS:
        cwords = set(re.findall(r"[a-z]{4,}", c["text"].lower()))
        overlap = len(qwords & cwords)
        if overlap:
            scored.append((overlap, c))
    scored.sort(key=lambda x: -x[0])
    return [c for _, c in scored[:k]]

# system prompt and disclaimer
RAG_SYSTEM_PROMPT = """You are an educational assistant that DEFINES brain-tumor MRI imaging terms for a
clinician. You are given the patient's measured values and RETRIEVED reference passages that define the
imaging concepts.

You describe ONLY what the imaging measurement IS and what the tissue LOOKS LIKE on MRI. You describe a
STATIC IMAGE. You must NEVER say what the tumor is DOING, what it MEANS clinically, or what will happen.
You ADD DEFINITIONS OR additional theoretical relevant information found in specific references. Cite the source: (SOURCE-it is found 
in the first line of each reference).

You must NEVER output: grade, type, aggressiveness; prognosis, survival, outcome, risk; treatment,
therapy, surgery; any word implying activity or change over time (growing, growth, spreading,
progressing, active, invasive, increased blood flow); any inference clause ("this suggests...",
"indicating that...", "consistent with..."); any clinical fact not in the retrieved passages.

When giving a definition, use the wording of the retrieved passage closely. Do NOT add your own
interpretation, elaboration, or clauses about what a finding "may indicate" or represent. State
the passage's definition and stop.

Describe the picture, define the term, cite the
source. End with the disclaimer.
"""
DISCLAIMER = ("Educational explanation of imaging findings only. Not a diagnosis, treatment recommendation or prediction of outcome.")

# light safety guard
_TYPE_NAMES = ["glioblastoma", "astrocytoma", "oligodendroglioma", "gbm"]

def _light_guard(text):
    out = text
    for t in _TYPE_NAMES:
        out = re.sub(r"\b" + re.escape(t) + r"\b", "the tumor", out, flags=re.IGNORECASE)

    # remove forbidden clauses
    forbidden_clauses = [
        r",?\s*(which |this )?(may |can )?(suggest|indicat|reflect|impl|represent)\w*[^.;:]*"
        r"(grow|growth|aggressi|invasi|spread|progress|active|angiogen|malignan|inflam)\w*[^.;:]*",
        r",?\s*(due to|caused by|because of|resulting from)[^.;:]*"
        r"(blood flow|vascular|perfusion|proliferat|angiogen)\w*[^.;:]*",
        r",?\s*(actively |rapidly )?(growing|growth)[^.;:]*",
    ]
    for pat in forbidden_clauses:
        out = re.sub(pat, "", out, flags=re.IGNORECASE)

    # clean up connectors
    out = re.sub(r"\b(due to|caused by|because of|which may|which can|resulting from)\s*[,.:;]",
                 ".", out, flags=re.IGNORECASE)
    out = re.sub(r"\s*,\s*\.", ".", out)
    out = re.sub(r"\s+([,.;:])", r"\1", out)
    out = re.sub(r"\.\s*\.", ".", out)
    out = re.sub(r"\s{2,}", " ", out)

    if "not a diagnosis" not in out.lower():
        out = out.rstrip() + "\n\n" + DISCLAIMER
    return out.strip()

# RAG entry point
def explain_with_rag(finding_query, findings, generate_fn, k=3):
    """Retrieve relevant passages and have the LLM explain, grounded in them."""
    import json
    hits = retrieve(finding_query, k=k)
    if not hits:
        raise RuntimeError("no reference passages retrieved")
    refs = "\n\n".join(f"[{h['source']}] {h['text']}" for h in hits)
    user = (f"Finding to explain: {finding_query}\n\n"
            f"Patient findings:\n{json.dumps(findings, indent=2)}\n\n"
            f"Retrieved references (use ONLY these):\n{refs}\n\n"
            f"End with this disclaimer exactly:\n{DISCLAIMER}")
    raw = generate_fn(RAG_SYSTEM_PROMPT, user)
    return _light_guard(raw)

def strip_identifiers(findings):
    """Remove patient identifiers before anything goes to the model."""
    f = copy.deepcopy(findings)
    f.pop("patient_id", None)
    f.pop("provenance", None)
    return f

## 3. Build the reference index

Load the curated open-access passages from `references/` and index them for retrieval. This runs once, the index is reused for every explanation.

In [7]:
built = build_index(references_dir)
print("FAISS semantic index built:", built)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS semantic index built: True


## 4. The local generator

A single `generate_fn(system, user)` that calls the local model via Ollama. Temperature 0 keeps it factual, the model only rephrases the retrieved passages, it does not invent.

In [8]:
def generate_fn(system, user):
    """Local model via Ollama. No data leaves the machine."""
    r = requests.post("http://localhost:11434/api/chat", json={
        "model": LOCAL_MODEL, "stream": False,
        "options": {"temperature": TEMPERATURE},
        "messages": [{"role": "system", "content": system},
                     {"role": "user", "content": user}],
    }, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.json()["message"]["content"].strip()

## 5. Build a query for a finding

Each finding becomes a short query that drives retrieval. This mirrors the application's clickable terms (necrosis, enhancing, margin, size) and the overall view.

In [9]:
def term_query(findings, term_key=None):
    """Turn a finding (or the overall report) into a retrieval query."""
    f = findings or {}
    if term_key == "necrosis_fraction":
        return f"necrosis level: {f.get('composition', {}).get('necrosis_category', '')}"
    if term_key == "enhancing_fraction_of_wt":
        return f"enhancing fraction: {f.get('composition', {}).get('enhancing_category', '')}"
    if term_key == "margin_descriptor":
        return f"margins: {f.get('morphology', {}).get('margin_descriptor', '')}"
    if term_key == "size_cm3":
        return f"tumour size: {f.get('size_cm3', {}).get('whole_tumour_cm3', '')} cm3"
    return "overall tumour composition and characteristics"

## 6. Load one patient and explain (overall)

Load a patient's findings, build the query, retrieve the relevant passages, and have the local model explain, grounded only in those passages, citing each source.

In [ ]:
finding_files = sorted(glob.glob(os.path.join(findings_dir, "*_findings.json")))
print(f"{len(finding_files)} findings files available.")
with open(finding_files[3]) as f:
    findings = json.load(f)
print("Explaining for:", findings["patient_id"])
context = strip_identifiers(findings)
query = term_query(findings, None)
try:
    overall = explain_with_rag(query, context, generate_fn, k=TOP_K)
    print(overall)
except Exception as e:
    print(f"\n*** RAG explanation failed: {e} ***")

## 7. Inspect what was retrieved

For transparency, show which passages the retriever returned for a query - this is the "grounding" the explanation is built on, and what makes each citation checkable.

In [12]:
hits = retrieve(term_query(findings, "margin_descriptor"), k=TOP_K)
for h in hits:
    print(f"[{h['source']}]  ({h['file']})")
    print(f"  {h['text'][:200]}...\n")

[Definitions]  (definitions.txt)
  Margin: A brain tumor margin is the boundary between a tumor and the surrounding brain tissue. Primary brain tumors rarely have sharp, distinct edges, so the margin is often difficult to define on ima...

[Definitions]  (definitions.txt)
  Edema: Cerebral edema is an excess accumulation of fluid in the intra- and extracellular spaces of the brain. On MRI it appears as high signal on T2-weighted and FLAIR images....

[StatPearls, Cerebral Edema 2023]  (StatPearls_Cerebral_Edema_2023.txt)
  Cerebral edema is an excess accumulation of fluid in the brain. On MRI it appears as high signal on T2-weighted and FLAIR images....



## End of notebook

All experiments completed. Results are saved.